# Morphological U-Net

In [ ]:
# init variables

DATA_SRC = "/kaggle/input/datasets/thanasisrigas/task02-heart/Task02_Heart"
%env TASK=Task02_Heart

In [ ]:
# repo and requirements

!rm -rf repo
!git clone "https://github.com/thnrigas/morph-unet.git" repo
%cd repo
!pip install -q medpy batchgenerators==0.21

In [ ]:
# setup

import torch, os, shutil
import train_eval, config

from datasets.two_dim.NumpyDataLoader import NumpyDataSet
from networks.UNET import UNet

y = UNet(num_classes=3, in_channels=1)(torch.rand(2, 1, 64, 64))

# copy DATA_SRC into config.DATA_DIR to be able to write to it
dst = config.DATA_DIR
os.makedirs(dst.parent, exist_ok=True)
if not dst.exists():
    shutil.copytree(DATA_SRC, dst)

In [ ]:
# preprocessing

!python run_preprocessing.py

In [ ]:
# baseline

for f in [0, 1, 2]:
    !python train_eval.py --tag baseline --num-workers 0 --fold {f}

In [ ]:
# staticbank

for f in [0, 1, 2]:
    !python train_eval.py --tag staticbank --static-auto --num-workers 0 --fold {f}

In [ ]:
# morphbank

for f in [0, 1, 2]:
    !python train_eval.py --tag morphbank --morph-bank auto --num-workers 0 --fold {f}

In [ ]:
# convctrl

for f in [0, 1, 2]:
    !python train_eval.py --tag convctrl --morph-bank auto --conv-control --num-workers 0 --fold {f}

In [ ]:
# results

for t in ['baseline', 'staticbank', 'morphbank', 'convctrl']:
    !python train_eval.py --fold-mean {t}

!python train_eval.py --compare \
    baseline_mean_scores.json staticbank_mean_scores.json \
    morphbank_mean_scores.json convctrl_mean_scores.json